<a href="https://colab.research.google.com/github/chjk86/hansi-database/blob/main/Untitled9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# =====================================================================
# [Step 0] 환경 설정 및 필수 패키지
# =====================================================================
!pip install -q transformers datasets scikit-learn accelerate

import os
import re
import glob
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# 방안 A: 난수 시드 변경 (Local Minimum 탈출)
RANDOM_SEED = 2026
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

# GitHub 저장소 클론
if not os.path.exists('/content/hansi-database'):
    !git clone https://github.com/chjk86/hansi-database.git /content/hansi-database

# =====================================================================
# [Step 1] 경로 설정 및 초고속 데이터 파싱 (정규식 배제)
# =====================================================================
FRONTIER_FILE = "/content/hansi-database/2026_DH_poster/변새시_2차정리본.txt"
MUNZIP_DIR = "/content/hansi-database/2025_munzip_title_text_ver"

def extract_pure_hanzi(text):
    clean_text = re.sub(r'<[^>]+>', '', text)
    return re.sub(r'[^\u4E00-\u9FFF]', '', clean_text)

# 정규식 대신 split을 활용한 초고속 태그 추출 함수
def fast_extract_tag_content(content, tag_name):
    extracted = []
    start_tag = f"<{tag_name}>"
    end_tag = f"</{tag_name}>"

    parts = content.split(start_tag)
    for part in parts[1:]:
        if end_tag in part:
            extracted.append(part.split(end_tag)[0])
    return extracted

frontier_data = []
frontier_texts = set()

print("[진행] 변새시(정답 데이터) 파싱을 시작합니다...")
with open(FRONTIER_FILE, 'r', encoding='utf-8') as f:
    content = f.read()

# 변새시 원문 추출 (빠른 파싱)
poem_texts = fast_extract_tag_content(content, "text")
for text_content in poem_texts:
    hanzi = extract_pure_hanzi(text_content)
    if hanzi and len(hanzi) >= 10:
        frontier_data.append({'text': hanzi, 'labels': 1})
        frontier_texts.add(hanzi)

TARGET_COUNT = len(frontier_data)
print(f" => 변새시 파싱 완료: {TARGET_COUNT}건")

non_frontier_pool = []
munzip_files = glob.glob(os.path.join(MUNZIP_DIR, "*.txt")) + glob.glob(os.path.join(MUNZIP_DIR, "*.xml"))
random.shuffle(munzip_files)

print(f"\n[진행] 대조군(비변새시) 초고속 데이터 수집을 시작합니다. (목표: {TARGET_COUNT}건)")
parsed_file_count = 0

for file_path in munzip_files:
    if len(non_frontier_pool) >= TARGET_COUNT * 2:
        break
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()

        munzip_texts = fast_extract_tag_content(content, "text")
        for text_content in munzip_texts:
            hanzi = extract_pure_hanzi(text_content)
            if len(hanzi) >= 10 and hanzi not in frontier_texts:
                non_frontier_pool.append({'text': hanzi, 'labels': 0})
    except Exception:
        pass

    parsed_file_count += 1
    if parsed_file_count % 100 == 0:
        print(f" - 문집 파일 스캔 중... ({parsed_file_count}/{len(munzip_files)}) | 현재 수집된 비변새시: {len(non_frontier_pool)}건")

random.shuffle(non_frontier_pool)
sampled_non_frontier = non_frontier_pool[:TARGET_COUNT]
df = pd.DataFrame(frontier_data + sampled_non_frontier)
print(f"\n[데이터 구성 완료] 총 {len(df)}수 준비됨.")

# =====================================================================
# [Step 2] 데이터 분할 (골드 테스트셋 고립)
# =====================================================================
train_val_df, test_df = train_test_split(df, test_size=0.15, random_state=RANDOM_SEED, stratify=df['labels'])
train_df_full, val_df = train_test_split(train_val_df, test_size=0.176, random_state=RANDOM_SEED, stratify=train_val_df['labels'])

MODEL_NAME = "ethanyt/guwenbert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=150)

tokenized_val = Dataset.from_pandas(val_df).map(tokenize_function, batched=True)
tokenized_test = Dataset.from_pandas(test_df).map(tokenize_function, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, average='binary', zero_division=0)
    }

# =====================================================================
# [Step 3] 학습 곡선(Learning Curve) 실험 루프
# =====================================================================
data_fractions = [0.1, 0.3, 0.5, 0.8, 1.0]
results = []

print("\n[학습 곡선 도출 실험 시작] 난수 시드 및 학습률 조정 적용됨")

for frac in data_fractions:
    print(f"\n========== 훈련 데이터 {int(frac*100)}% 투입 실험 ==========")

    if frac == 1.0:
        current_train_df = train_df_full
    else:
        current_train_df, _ = train_test_split(train_df_full, train_size=frac, random_state=RANDOM_SEED, stratify=train_df_full['labels'])

    tokenized_train = Dataset.from_pandas(current_train_df).map(tokenize_function, batched=True)

    # 모델 매번 초기화 (데이터 누출 및 가중치 누적 방지)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    # 방안 B: 학습률(learning_rate) 하향 조정 (3e-5 -> 2e-5)
    training_args = TrainingArguments(
        output_dir=f"/content/checkpoints_{int(frac*100)}",
        eval_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
        save_strategy="no",
        seed=RANDOM_SEED
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # 평가 (고립된 test_df 대상)
    test_results = trainer.predict(tokenized_test).metrics
    f1 = test_results['test_f1']
    precision = test_results['test_precision']
    recall = test_results['test_recall']

    print(f" => [평가 결과] {int(frac*100)}% 데이터 | F1: {f1:.4f} | 정밀도: {precision:.4f} | 재현율: {recall:.4f}")
    results.append({
        'Data Size (%)': int(frac * 100),
        'F1 Score': f1,
        'Precision': precision,
        'Recall': recall
    })

# =====================================================================
# [Step 4] 결과 시각화 및 저장
# =====================================================================
results_df = pd.DataFrame(results)
csv_output = '/content/bert_learning_curve.csv'
results_df.to_csv(csv_output, index=False)

plt.figure(figsize=(10, 6), dpi=300)
plt.plot(results_df['Data Size (%)'], results_df['F1 Score'], marker='o', linestyle='-', linewidth=2, label='F1 Score')
plt.plot(results_df['Data Size (%)'], results_df['Precision'], marker='s', linestyle='--', linewidth=2, label='Precision')
plt.plot(results_df['Data Size (%)'], results_df['Recall'], marker='^', linestyle='-.', linewidth=2, label='Recall')

for i, row in results_df.iterrows():
    plt.text(row['Data Size (%)'], row['F1 Score'] + 0.01, f"{row['F1 Score']:.3f}", ha='center', va='bottom', fontweight='bold', color='C0')

plt.title('BERT Fine-tuning Performance by Training Data Size', fontsize=16, pad=15)
plt.xlabel('Training Data Size (%)', fontsize=14)
plt.ylabel('Score', fontsize=14)
plt.xticks(results_df['Data Size (%)'])
plt.ylim(0.4, 1.05)
plt.grid(True, alpha=0.6)
plt.legend(loc='lower right', fontsize=12)
plt.tight_layout()

png_output = '/content/bert_learning_curve.png'
plt.savefig(png_output)
plt.close()

files.download(csv_output)
files.download(png_output)
print("\n[실험 완료] 데이터 크기별 성능 결과 파일과 그래프가 저장되었습니다.")

[진행] 변새시(정답 데이터) 파싱을 시작합니다...
 => 변새시 파싱 완료: 259건

[진행] 대조군(비변새시) 초고속 데이터 수집을 시작합니다. (목표: 259건)

[데이터 구성 완료] 총 518수 준비됨.


config.json:   0%|          | 0.00/519 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.5k [00:00<?, ?B/s]

Map:   0%|          | 0/78 [00:00<?, ? examples/s]

Map:   0%|          | 0/78 [00:00<?, ? examples/s]

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


[실험 완료] 데이터 크기별 성능 결과 파일과 그래프가 저장되었습니다.
